# NearDup of text data

In [1]:
import os
import datasets

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())

True

### load data and view statistics

In [2]:
ds = datasets.load_dataset('datalama/kbl-simple', split='train')

In [3]:
from textanalyzer.processors import TokenCountProcessor

In [4]:
tc_proc = TokenCountProcessor()
text_count_ds = ds.map(tc_proc, batched=True, fn_kwargs={'text_column': 'sentence'})

In [5]:
token_stats = tc_proc.compute_stats(text_count_ds['token_count'])
print(f"Number of tokens: {tc_proc.format_token_counts(token_stats['total_tokens'])}")
token_stats

Number of tokens: 46.9M


{'total_tokens': 46918365,
 'avg_sentence_length': 109.43339654195209,
 'sentence_length_std': 122.02358913496003,
 'max_sentence_length': 8386,
 'min_sentence_length': 2,
 'median_sentence_length': 78}

### MinHash (index-step)

In [6]:
from textanalyzer.processors.dedup import MinHashProcessor

In [7]:
min_hash_proc = MinHashProcessor() # exact_substr paper's neardup param.

* process indexing 1% of data (takes too much time)

In [8]:
indexed_ds = ds.map(min_hash_proc, batched=True, num_proc=os.cpu_count(),
                                                   fn_kwargs={'text_column':'sentence'}, load_from_cache_file=False)

Map (num_proc=16):   0%|          | 0/428739 [00:00<?, ? examples/s]

In [9]:
indexed_ds.save_to_disk("/Users/dongwook/workspace/open_source/klb_simple_minhashed")

### LSH (Dedup step)

In [9]:
from textanalyzer.processors.dedup import StreamingDedupProcessor

In [10]:
stream_proc = StreamingDedupProcessor()

In [11]:
batch_indexed_ds = indexed_ds.map(stream_proc, batched=True, with_indices=True)

Map:   0%|          | 0/428739 [00:00<?, ? examples/s]

In [12]:
dup_indices = [i for i, is_dup in enumerate(batch_indexed_ds['is_dups']) if is_dup]

In [17]:
dedup_indices = [i for i, is_dup in enumerate(batch_indexed_ds['is_dups']) if not is_dup]

In [19]:
dedup_sentences = [sent for i, sent in enumerate(batch_indexed_ds['sentence']) if i in dedup_indices]

---

### MinHash params

In [6]:
from datasketch import MinHashLSH

# Parameters
num_perm = 800
threshold = 0.91

# Create an LSH index with a threshold
lsh = MinHashLSH(threshold=threshold, num_perm=num_perm)

# Get band and row numbers
bands = lsh.b
rows = lsh.r

print(f"Number of bands: {bands}")
print(f"Number of rows per band: {rows}")

Number of bands: 20
Number of rows per band: 40
